# Day 2 Machine Translation: Luganda to English


Language pair: Luganda to English
Dataset: [`AI-Lab-Makerere/en_lg`](https://huggingface.co/AI-Lab-Makerere/en_lg) — official Makerere AI Lab parallel corpus
Framework: TensorFlow / Keras

Models trained:
1. Statistical baseline (IBM Model 1 word alignment via NLTK)
2. Seq2seq LSTM with attention, trained from scratch (TensorFlow/Keras)
3. Fine-tuned pretrained model (Helsinki-NLP OPUS-MT, TensorFlow)

Evaluation: BLEU score (sacrebleu) on a shared held-out test split, same for all three models.


In [ ]:

pip install tensorflow transformers datasets sacrebleu nltk pandas numpy scikit-learn -q


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))  # expect empty on CPU-only setup


In [ ]:
from datasets import load_dataset

raw = load_dataset("AI-Lab-Makerere/en_lg")
print(raw)
raw["train"][0]


In [ ]:
# Converting to a flat DataFrame of (english, luganda) 

def to_df(split):
    data = raw[split]
    pairs = [(ex["translation"]["en"], ex["translation"]["lg"]) for ex in data]
    return pd.DataFrame(pairs, columns=["en", "lg"])

df = to_df("train")
print(df.shape)
df.head()


In [ ]:

df = df.dropna().drop_duplicates()
df["en"] = df["en"].str.strip()
df["lg"] = df["lg"].str.strip()
df = df[(df["en"].str.len() > 0) & (df["lg"].str.len() > 0)]

df["en_len"] = df["en"].str.split().apply(len)
df["lg_len"] = df["lg"].str.split().apply(len)

print(df[["en_len", "lg_len"]].describe())


MAX_LEN = 25
df = df[(df["en_len"] <= MAX_LEN) & (df["lg_len"] <= MAX_LEN)].reset_index(drop=True)
print("After length filtering:", df.shape)


In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.1, random_state=SEED)
train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=SEED)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

test_df.to_csv("test_split.csv", index=False)


In [ ]:
import nltk
nltk.download("punkt", quiet=True)
from nltk.translate import IBMModel1, AlignedSent

bitext = [AlignedSent(row.lg.split(), row.en.split()) for row in train_df.itertuples()]
ibm1 = IBMModel1(bitext, 5)  

def ibm1_translate(lg_sentence):
    words = lg_sentence.split()
    translated = []
    for w in words:
        probs = {en_w: ibm1.translation_table[w].get(en_w, 0) for en_w in ibm1.translation_table[w]}
        best = max(probs, key=probs.get) if probs else w
        translated.append(best)
    return " ".join(translated)


sample = test_df.iloc[0]
print("LG :", sample.lg)
print("REF:", sample.en)
print("HYP:", ibm1_translate(sample.lg))


In [ ]:
baseline_hyps = [ibm1_translate(lg) for lg in test_df["lg"]]


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

VOCAB_SIZE = 8000
MAX_SEQ_LEN = MAX_LEN + 2  

def add_tokens(s):
    return f"<start> {s} <end>"

train_lg = train_df["lg"].apply(add_tokens).tolist()
train_en = train_df["en"].apply(add_tokens).tolist()

src_tok = Tokenizer(num_words=VOCAB_SIZE, filters='', oov_token="<unk>")
src_tok.fit_on_texts(train_lg)
tgt_tok = Tokenizer(num_words=VOCAB_SIZE, filters='', oov_token="<unk>")
tgt_tok.fit_on_texts(train_en)

def encode(tok, texts):
    seqs = tok.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_SEQ_LEN, padding="post")

X_train = encode(src_tok, train_lg)
y_train = encode(tgt_tok, train_en)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("Source vocab:", min(VOCAB_SIZE, len(src_tok.word_index)+1))
print("Target vocab:", min(VOCAB_SIZE, len(tgt_tok.word_index)+1))


In [ ]:
EMBED_DIM = 128
UNITS = 256
SRC_VOCAB = min(VOCAB_SIZE, len(src_tok.word_index) + 1)
TGT_VOCAB = min(VOCAB_SIZE, len(tgt_tok.word_index) + 1)

class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embed_dim, units):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embed_dim, mask_zero=True)
        self.lstm = tf.keras.layers.LSTM(units, return_sequences=True, return_state=True)

    def call(self, x):
        x = self.embedding(x)
        out, h, c = self.lstm(x)
        return out, h, c

class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, query, values):
        query_exp = tf.expand_dims(query, 1)
        score = self.V(tf.nn.tanh(self.W1(values) + self.W2(query_exp)))
        weights = tf.nn.softmax(score, axis=1)
        context = tf.reduce_sum(weights * values, axis=1)
        return context, weights

class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embed_dim, units):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embed_dim, mask_zero=True)
        self.lstm = tf.keras.layers.LSTM(units, return_sequences=True, return_state=True)
        self.attention = BahdanauAttention(units)
        self.fc = tf.keras.layers.Dense(vocab_size)

    def call(self, x, hidden, enc_output):
        context, _ = self.attention(hidden, enc_output)
        x = self.embedding(x)
        x = tf.concat([tf.expand_dims(context, 1), x], axis=-1)
        out, h, c = self.lstm(x)
        out = tf.reshape(out, (-1, out.shape[2]))
        out = self.fc(out)
        return out, h, c

encoder = Encoder(SRC_VOCAB, EMBED_DIM, UNITS)
decoder = Decoder(TGT_VOCAB, EMBED_DIM, UNITS)
optimizer = tf.keras.optimizers.Adam()
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

def loss_fn(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_object(real, pred)
    mask = tf.cast(mask, dtype=loss_.dtype)
    return tf.reduce_mean(loss_ * mask)


In [ ]:
BATCH_SIZE = 32
EPOCHS = 10  

dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(len(X_train)).batch(BATCH_SIZE, drop_remainder=True)

@tf.function
def train_step(src, tgt):
    loss = 0
    with tf.GradientTape() as tape:
        enc_out, enc_h, enc_c = encoder(src)
        dec_h = enc_h
        dec_input = tf.expand_dims(tgt[:, 0], 1)
        for t in range(1, tgt.shape[1]):
            preds, dec_h, _ = decoder(dec_input, dec_h, enc_out)
            loss += loss_fn(tgt[:, t], preds)
            dec_input = tf.expand_dims(tgt[:, t], 1)  
    batch_loss = loss / int(tgt.shape[1])
    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))
    return batch_loss

for epoch in range(EPOCHS):
    total_loss = 0
    steps = 0
    for src_batch, tgt_batch in dataset:
        total_loss += train_step(src_batch, tgt_batch)
        steps += 1
    print(f"Epoch {epoch+1}/{EPOCHS} — loss: {total_loss/steps:.4f}")


In [ ]:
def seq2seq_translate(sentence, max_len=MAX_SEQ_LEN):
    sentence = add_tokens(sentence)
    seq = src_tok.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=MAX_SEQ_LEN, padding="post")
    enc_out, enc_h, enc_c = encoder(seq)
    dec_h = enc_h
    dec_input = tf.expand_dims([tgt_tok.word_index.get("<start>", 1)], 0)
    result = []
    for _ in range(max_len):
        preds, dec_h, _ = decoder(dec_input, dec_h, enc_out)
        pred_id = tf.argmax(preds[0]).numpy()
        word = tgt_tok.index_word.get(pred_id, "")
        if word == "<end>" or word == "":
            break
        result.append(word)
        dec_input = tf.expand_dims([pred_id], 0)
    return " ".join(result)

# Sanity check
print(seq2seq_translate(test_df.iloc[0].lg))


In [ ]:
seq2seq_hyps = [seq2seq_translate(lg) for lg in test_df["lg"]]


In [ ]:
from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM

# Switched to the dedicated Luganda->English bilingual checkpoint so this model's
# direction matches the other two (baseline + seq2seq), which are both LG->EN.
MODEL_CKPT = "Helsinki-NLP/opus-mt-lg-en"

hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)
hf_model = TFAutoModelForSeq2SeqLM.from_pretrained(MODEL_CKPT)


In [ ]:

FT_SUBSET = 2000
ft_train = train_df.sample(min(FT_SUBSET, len(train_df)), random_state=SEED)

def encode_batch(df_subset):
    # Source is now Luganda (lg), target/labels is English (en) -- matches the
    # LG->EN direction used by the baseline and seq2seq models above.
    inputs = hf_tokenizer(list(df_subset["lg"]), padding=True, truncation=True, max_length=MAX_LEN, return_tensors="tf")
    with hf_tokenizer.as_target_tokenizer():
        labels = hf_tokenizer(list(df_subset["en"]), padding=True, truncation=True, max_length=MAX_LEN, return_tensors="tf")
    return inputs, labels["input_ids"]

train_inputs, train_labels = encode_batch(ft_train)

hf_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5))
hf_model.fit(
    x={"input_ids": train_inputs["input_ids"], "attention_mask": train_inputs["attention_mask"]},
    y=train_labels,
    batch_size=4,
    epochs=2,
)


In [ ]:
def hf_translate(sentence):
    inputs = hf_tokenizer(sentence, return_tensors="tf", truncation=True, max_length=MAX_LEN)
    out = hf_model.generate(**inputs, max_length=MAX_LEN)
    return hf_tokenizer.decode(out[0], skip_special_tokens=True)

# Sanity check -- now translating FROM Luganda, same as the other two models
print(hf_translate(test_df.iloc[0].lg))


In [ ]:
hf_hyps = [hf_translate(lg) for lg in test_df["lg"]]


In [ ]:
import sacrebleu

# All three models are now LG->EN, so all three are scored against the same
# English references.
refs_en = [[r] for r in test_df["en"].tolist()]

bleu_baseline   = sacrebleu.corpus_bleu(baseline_hyps, refs_en)
bleu_seq2seq    = sacrebleu.corpus_bleu(seq2seq_hyps, refs_en)
bleu_pretrained = sacrebleu.corpus_bleu(hf_hyps, refs_en)

# chrF++ is more forgiving of morphological variation, which matters here since
# Luganda is agglutinative and BLEU's exact n-gram matching can unfairly punish
# outputs that are morphologically close but not identical. Character n-grams
# (chrF) don't require exact word-boundary matches, so a translation that gets
# the root/stem right but differs on a noun-class prefix or affix still scores
# reasonably instead of losing full credit for that word.
chrf_baseline   = sacrebleu.corpus_chrf(baseline_hyps, refs_en, word_order=2)
chrf_seq2seq    = sacrebleu.corpus_chrf(seq2seq_hyps, refs_en, word_order=2)
chrf_pretrained = sacrebleu.corpus_chrf(hf_hyps, refs_en, word_order=2)

results = pd.DataFrame({
    "Model": ["IBM Model 1 (baseline)", "Seq2seq LSTM+Attention (scratch)", "Fine-tuned OPUS-MT (lg-en)"],
    "Direction": ["LG->EN", "LG->EN", "LG->EN"],
    "BLEU": [bleu_baseline.score, bleu_seq2seq.score, bleu_pretrained.score],
    "chrF++": [chrf_baseline.score, chrf_seq2seq.score, chrf_pretrained.score],
})
results


#  Notes for the Report

Dataset: `AI-Lab-Makerere/en_lg`, Makerere AI Lab English to Luganda parallel corpus -- cite as local-language data.
Preprocessing: dropped duplicates/empties, capped sequence length at `MAX_LEN` tokens for CPU feasibility.
Direction: all three models are evaluated LG->EN for a fair, apples-to-apples comparison.
  The pretrained model uses `Helsinki-NLP/opus-mt-lg-en`, a dedicated bilingual checkpoint,
  fine-tuned further on the LG->EN training split (rather than the multilingual en-mul
  checkpoint, which was pointed the wrong direction and wasn't guaranteed to cover Luganda well).
Metrics: BLEU (sacrebleu) plus chrF++ (character n-gram F-score, word_order=2). chrF++ is
  included because Luganda is agglutinative (noun-class prefixes, rich morphology), and
  BLEU's exact word-level n-gram matching can unfairly penalize outputs that are morphologically
  close but not identical. chrF++ operates on character n-grams, so it credits partial
  word-level matches more gracefully -- a useful cross-check against BLEU here.
Limitation to flag in the report: only one reference translation per source sentence, so
  scores may be somewhat pessimistic relative to a multi-reference setup.
Discussion to include: which model scored highest on BLEU vs chrF++, and whether the two
  metrics agree; pretrained/fine-tuned models typically transfer better even on low-resource
  pairs due to multilingual pretraining; the from-scratch seq2seq is data-hungry; the
  statistical baseline is a weak floor.
Keep the report to 3 pages max -- dataset, models/hyperparameters, results table, and a short discussion.
